# Silver: Supplier Deduplication
Removes duplicate supplier records introduced by SAP case-variant exports.

In [ ]:
from pyspark.sql.functions import upper, trim, row_number, col, current_timestamp, lit
from pyspark.sql.window import Window

df_sup = spark.table('bronze_lakehouse.dim_suppliers_raw')
print(f'Input rows: {df_sup.count()}')

# Normalize supplier name for comparison
df_norm = df_sup.withColumn('_name_key', upper(trim(col('SupplierName'))))

# Keep highest SupplierID per normalized name (most recent entry wins)
window = Window.partitionBy('_name_key').orderBy(col('SupplierID').desc())
df_deduped = (df_norm
    .withColumn('_rank', row_number().over(window))
    .filter(col('_rank') == 1)
    .drop('_rank', '_name_key')
    .withColumn('_cleansed_at', current_timestamp())
    .withColumn('_layer', lit('silver')))

removed = df_sup.count() - df_deduped.count()
print(f'Removed {removed} duplicate records')
print(f'Output rows: {df_deduped.count()}')

df_deduped.write.format('delta').mode('overwrite').saveAsTable('silver_lakehouse.dim_suppliers')
print('✅ Written to silver_lakehouse.dim_suppliers')